# Text Generation and Reasoning with LLMs

Exploring decoding algorithms, few-shot prompting, and ReAct agents for language models.

---
## Section 1: Decoding Algorithms

Implementing and analyzing various text generation strategies.

In [10]:
!pip install -q vllm transformers datasets evaluate

In [11]:
"""Set device and random seeds"""

from tqdm import tqdm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}')

def set_seed(seed=19260817):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

In [12]:
from datasets import load_dataset

dataset = load_dataset('igormorgado/ROCStories2016')
train_data, dev_data, test_data = dataset['train'], dataset['validation'], dataset['test']

# Construct 'prompt' field from sentence1 for story generation
def add_prompt(example):
    example['prompt'] = example['sentence1']
    return example

train_data = train_data.map(add_prompt)
dev_data = dev_data.map(add_prompt)
test_data = test_data.map(add_prompt)

print(train_data[0])

In [13]:
"""Prepare evaluation metrics"""

from transformers import RobertaForSequenceClassification, RobertaTokenizer

cola_model_name = "textattack/roberta-base-CoLA"
cola_tokenizer = RobertaTokenizer.from_pretrained(cola_model_name)
cola_model = RobertaForSequenceClassification.from_pretrained(cola_model_name).to(device)

def batchify(data, batch_size):
    assert batch_size > 0
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]

In [14]:
"""Evaluation functions"""

from transformers import GPT2LMHeadModel, GPT2TokenizerFast

_ppl_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
_ppl_model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
_ppl_model.eval()

def compute_perplexity(texts, batch_size=4, max_length=512):
    """Compute mean perplexity of a list of texts using GPT-2."""
    nlls = []
    for text in texts:
        if not text.strip():
            continue
        encodings = _ppl_tokenizer(text, return_tensors='pt', truncation=True, max_length=max_length).to(device)
        input_ids = encodings.input_ids
        if input_ids.shape[1] < 2:
            continue
        with torch.no_grad():
            outputs = _ppl_model(input_ids, labels=input_ids)
        nll = outputs.loss.item()
        nlls.append(nll)
    if not nlls:
        return float('inf')
    import math
    return math.exp(sum(nlls) / len(nlls))


def compute_fluency(texts, batch_size=8):
    scores = []
    for b_texts in batchify(texts, batch_size):
        inputs = cola_tokenizer(b_texts, padding=True, truncation=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = cola_model(**inputs).logits
        predictions = torch.argmax(logits, dim=-1)
        scores.extend(predictions.cpu().tolist())
    return sum(scores) / len(scores)


def compute_diversity(texts, n=3):
    all_ngrams = []
    for text in texts:
        tokens = text.split()
        ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        all_ngrams.extend(ngrams)
    if len(all_ngrams) == 0:
        return 0
    return len(set(all_ngrams)) / len(all_ngrams)


def compute_repetition(texts, n=4):
    total = 0
    repeated = 0
    for text in texts:
        tokens = text.split()
        ngrams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
        total += len(ngrams)
        repeated += len(ngrams) - len(set(ngrams))
    if total == 0:
        return 0
    return repeated / total

In [15]:
"""Load model and tokenizer"""

from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = 'gpt2'
tokenizer = GPT2Tokenizer.from_pretrained(model_name, pad_token="<|endoftext|>")
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.eval()

In [16]:
def decode(prompts, max_len, method, **kwargs):
    encodings_dict = tokenizer(prompts, return_tensors="pt", padding=True)
    input_ids = encodings_dict['input_ids'].to(device)
    attention_mask = encodings_dict['attention_mask'].to(device)

    batch_size, input_seq_len = input_ids.shape
    unfinished_sequences = torch.ones(batch_size, dtype=torch.long, device=device)
    eos_token_id_tensor = torch.tensor([tokenizer.eos_token_id]).to(device)

    for step in range(max_len):
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        next_token_logits = outputs.logits[:, -1, :]
        next_tokens = method(next_token_logits, **kwargs)
        next_tokens = next_tokens * unfinished_sequences + tokenizer.pad_token_id * (1 - unfinished_sequences)

        input_ids = torch.cat([input_ids, next_tokens[:, None]], dim=-1)
        attention_mask = torch.cat(
            [attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=-1
        )
        unfinished_sequences = unfinished_sequences.mul(
            next_tokens.tile(eos_token_id_tensor.shape[0], 1).ne(eos_token_id_tensor.unsqueeze(1)).prod(dim=0)
        )
        if unfinished_sequences.max() == 0:
            break

    decoded = tokenizer.batch_decode(input_ids[:, input_seq_len:], skip_special_tokens=True)
    return decoded

In [17]:
"""Debug helper"""

dev_prompt = dev_data[0]['prompt']
print(f'Dev prompt: {dev_prompt}')
print()

def show_generations(method_name, method, n=3, max_len=100, **kwargs):
    """Generate n continuations from the dev prompt and display them."""
    for i in range(n):
        output = decode([dev_prompt], max_len, method, **kwargs)
        print(f'[{i+1}] {output[0][:200]}')
    print()

### 1.1 Greedy Decoding

Select the next token as the one with the highest logit. Implement the `greedy()` function.

- Input: `next_token_logits` - Tensor of shape `(B, V)` (batch size x vocabulary size)
- Output: `next_tokens` - LongTensor of shape `(B,)`

In [18]:
def greedy(next_token_logits):
    '''
    inputs:
    - next_token_logits: Tensor(size = (B, V), dtype = float)
    outputs:
    - next_tokens: Tensor(size = (B,), dtype = long)
    '''
    return torch.argmax(next_token_logits, dim=-1)

show_generations('Greedy', greedy)

### 1.2 Vanilla Sampling and Temperature Sampling

**Vanilla sampling**: Sample the next token from the probability distribution defined by softmax over logits.

**Temperature sampling**: Divide logits by temperature `t` before applying softmax. Implement both `sample()` and `temperature()` functions.

- For testing, we use `t = 0.8`, but your implementation should support arbitrary `t > 0`.

In [19]:
def sample(next_token_logits):
    '''
    inputs:
    - next_token_logits: Tensor(size = (B, V), dtype = float)
    outputs:
    - next_tokens: Tensor(size = (B,), dtype = long)
    Hint: use torch.multinomial()
    '''
    probs = F.softmax(next_token_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

show_generations('Vanilla Sampling', sample)

In [20]:
def temperature(next_token_logits, t):
    '''
    inputs:
    - next_token_logits: Tensor(size = (B, V), dtype = float)
    - t: float, temperature parameter (t > 0)
    outputs:
    - next_tokens: Tensor(size = (B,), dtype = long)
    '''
    scaled_logits = next_token_logits / t
    probs = F.softmax(scaled_logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

show_generations('Temperature (t=0.8)', temperature, t=0.8)

### 1.3 Top-k Sampling

Sample from only the top-k highest-probability tokens. The sampling probability should be proportional to the original logits, renormalized to sum to 1.

- For testing, we use `k = 20`, but your implementation should support arbitrary `k`.

In [21]:
def topk(next_token_logits, k):
    '''
    inputs:
    - next_token_logits: Tensor(size = (B, V), dtype = float)
    - k: int, number of top tokens to consider
    outputs:
    - next_tokens: Tensor(size = (B,), dtype = long)
    '''
    top_k_values, _ = torch.topk(next_token_logits, k, dim=-1)
    threshold = top_k_values[:, -1].unsqueeze(-1)
    filtered = next_token_logits.masked_fill(next_token_logits < threshold, float('-inf'))
    probs = F.softmax(filtered, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

show_generations('Top-k (k=20)', topk, k=20)

### 1.4 Top-p (Nucleus) Sampling

Sample from the smallest set of tokens whose cumulative probability exceeds threshold `p`.

- For testing, we use `p = 0.7`, but your implementation should support arbitrary `p`.

In [22]:
def topp(next_token_logits, p):
    '''
    inputs:
    - next_token_logits: Tensor(size = (B, V), dtype = float)
    - p: float, cumulative probability threshold (0 <= p <= 1)
    outputs:
    - next_tokens: Tensor(size = (B,), dtype = long)
    '''
    probs = F.softmax(next_token_logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, dim=-1, descending=True)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    # Remove tokens once cumulative prob exceeds p (keep at least 1 token)
    sorted_mask = (cumulative_probs - sorted_probs) >= p
    sorted_probs[sorted_mask] = 0.0
    # Scatter back to original ordering
    filtered_probs = torch.zeros_like(probs).scatter_(-1, sorted_indices, sorted_probs)
    return torch.multinomial(filtered_probs, num_samples=1).squeeze(-1)

show_generations('Top-p (p=0.7)', topp, p=0.7)

### 1.5 Evaluation and Analysis

Run the evaluation cell below, then complete the analysis tasks.

In [55]:
"""Evaluation"""

prompts = [item['prompt'] for item in test_data][:5]
GENERATIONS_PER_PROMPT = 5
MAX_LEN = 100

methods = {
    'greedy': {'method': greedy},
    'sample': {'method': sample},
    'temperature_0.8': {'method': temperature, 't': 0.8},
    'topk_20': {'method': topk, 'k': 20},
    'topp_0.7': {'method': topp, 'p': 0.7},
}

results = {}
for name, config in methods.items():
    all_texts = []
    for prompt in tqdm(prompts, desc=name):
        for _ in range(GENERATIONS_PER_PROMPT):
            texts = decode([prompt], MAX_LEN, **config)
            all_texts.extend(texts)
    
    ppl = compute_perplexity(all_texts)
    flu = compute_fluency(all_texts)
    div = compute_diversity(all_texts)
    rep = compute_repetition(all_texts)
    results[name] = {'perplexity': ppl, 'fluency': flu, 'diversity': div, 'repetition': rep}
    print(f'{name}: PPL={ppl:.2f}, Fluency={flu:.4f}, Diversity={div:.4f}, Repetition={rep:.4f}')

### 1.6 Temperature Sweep Analysis

temperature sampling with `t` values `[0.3, 0.5, 0.8, 1.0, 1.5]`. For each value, compute perplexity, diversity, and repetition rate. **Plot curves** showing how each metric changes with temperature.

3 test prompts with 5 generations each.

In [24]:
# TODO: Run temperature sweep and plot the curves
# For each t in [0.3, 0.5, 0.8, 1.0, 1.5]:
#   - Generate texts using temperature sampling (use 3 prompts, 5 generations each)
#   - Compute perplexity, diversity, repetition
# Plot 3 subplots: (1) t vs perplexity, (2) t vs diversity, (3) t vs repetition

temperatures = [0.3, 0.5, 0.8, 1.0, 1.5]
sweep_prompts = [item['prompt'] for item in test_data][:3]
SWEEP_GENS = 5

sweep_results = {'ppl': [], 'div': [], 'rep': []}

for t in temperatures:
    all_texts = []
    for prompt in tqdm(sweep_prompts, desc=f't={t}'):
        for _ in range(SWEEP_GENS):
            texts = decode([prompt], MAX_LEN, temperature, t=t)
            all_texts.extend(texts)
    sweep_results['ppl'].append(compute_perplexity(all_texts))
    sweep_results['div'].append(compute_diversity(all_texts))
    sweep_results['rep'].append(compute_repetition(all_texts))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(temperatures, sweep_results['ppl'], marker='o')
axes[0].set_title('Perplexity vs Temperature')
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Perplexity')

axes[1].plot(temperatures, sweep_results['div'], marker='o', color='green')
axes[1].set_title('Diversity vs Temperature')
axes[1].set_xlabel('Temperature')
axes[1].set_ylabel('Diversity (unique 3-grams ratio)')

axes[2].plot(temperatures, sweep_results['rep'], marker='o', color='red')
axes[2].set_title('Repetition vs Temperature')
axes[2].set_xlabel('Temperature')
axes[2].set_ylabel('Repetition (repeated 4-grams ratio)')

plt.tight_layout()
plt.show()

for t, p, d, r in zip(temperatures, sweep_results['ppl'], sweep_results['div'], sweep_results['rep']):
    print(f't={t}: PPL={p:.2f}, Diversity={d:.4f}, Repetition={r:.4f}')

In [27]:
# Run this in the FIRST cell before importing vllm
import os
import gc
import torch

# Make sure conda compiler is visible to Triton/vLLM worker processes
conda_prefix = os.environ.get("CONDA_PREFIX", "")
if conda_prefix:
    os.environ["CC"] = f"{conda_prefix}/bin/x86_64-conda-linux-gnu-gcc"
    os.environ["CXX"] = f"{conda_prefix}/bin/x86_64-conda-linux-gnu-g++"
    os.environ["PATH"] = f"{conda_prefix}/bin:" + os.environ["PATH"]

# Optional: more verbose vLLM logs
os.environ["VLLM_LOGGING_LEVEL"] = "WARNING"

# If any old models are in memory, clear them
for name in ["model", "_ppl_model", "cola_model", "llm"]:
    if name in globals():
        del globals()[name]
gc.collect()
torch.cuda.empty_cache()

print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

---
## Section 2: Prompting Strategies for Math Reasoning 

In [28]:
import os
# Use default HF cache (~/.cache/huggingface) - no need to override
# os.environ["HF_HOME"] = "..."

from vllm import LLM, SamplingParams
model_id = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
llm = LLM(model=model_id, enforce_eager=True)

In [30]:
from openai import OpenAI
from transformers import AutoTokenizer

class VLLMClient:
    def __init__(self, model_id, **kwargs):
        self.model_id = model_id

    def __call__(self, prompt: str, **kwargs):
        response = llm.generate(
            prompts=prompt,
            sampling_params=SamplingParams(
                temperature=kwargs.get("temperature", 0.2),
                max_tokens=kwargs.get("max_tokens", 256),
            )
        )
        return response[0].outputs[0].text

model_llm = VLLMClient(model_id)
model_llm("San Francisco is a", max_tokens=42)

In [31]:
ARC_EXAMPLARS = [
    {
        "question": "A ball is thrown straight up into the air. At the highest point, the ball's velocity is",
        "choices": ["zero", "at its maximum", "equal to the initial velocity", "negative"],
        "cot_answer": "When a ball is thrown straight up, it decelerates due to gravity. At the highest point, the ball momentarily stops before falling back down. Therefore, the velocity at the highest point is zero. The answer is 0.",
        "short_answer": "0"
    },
    {
        "question": "Which of the following is the best example of a chemical change?",
        "choices": ["crushing a rock", "melting butter", "burning wood", "dissolving sugar in water"],
        "cot_answer": "Crushing a rock, melting butter, and dissolving sugar are physical changes because the substance's chemical composition doesn't change. Burning wood is a chemical change because it produces new substances (ash, carbon dioxide, water vapor). The answer is 2.",
        "short_answer": "2"
    },
    {
        "question": "A student wants to determine how the mass of an object affects the force needed to move it. Which tool should the student use to measure mass?",
        "choices": ["ruler", "balance", "thermometer", "graduated cylinder"],
        "cot_answer": "A ruler measures length, a thermometer measures temperature, and a graduated cylinder measures volume. A balance is used to measure mass by comparing the unknown mass to known masses. The answer is 1.",
        "short_answer": "1"
    },
    {
        "question": "Earth's core is primarily made up of which of the following materials?",
        "choices": ["ite andite", "ite and silica", "iron and nickel", "ite andite"],
        "cot_answer": "Scientists have determined through seismic studies and analysis of meteorites that Earth's core is primarily composed of iron and nickel. These heavy elements sank to the center during Earth's formation. The answer is 2.",
        "short_answer": "2"
    },
    {
        "question": "Which of these is a function of the skeletal system?",
        "choices": ["to transport oxygen", "to protect organs", "to digest food", "to regulate body temperature"],
        "cot_answer": "Transporting oxygen is the circulatory system's function, digesting food is the digestive system's, and regulating body temperature involves the integumentary and circulatory systems. Protecting organs (like the brain by the skull, heart by the ribcage) is a key function of the skeletal system. The answer is 1.",
        "short_answer": "1"
    },
    {
        "question": "What happens to the resistance in a wire as the temperature increases?",
        "choices": ["resistance decreases", "resistance increases", "resistance stays the same", "resistance becomes zero"],
        "cot_answer": "In most conductors (like metals), increasing temperature causes atoms to vibrate more, which increases collisions with electrons flowing through the wire. This increased collision rate means higher resistance. The answer is 1.",
        "short_answer": "1"
    },
    {
        "question": "A plant wilts when it does not receive enough",
        "choices": ["carbon dioxide", "water", "oxygen", "nitrogen"],
        "cot_answer": "Wilting occurs when plant cells lose turgor pressure due to water loss. While plants need carbon dioxide for photosynthesis, oxygen for respiration, and nitrogen for growth, it is the lack of water that directly causes wilting by reducing turgor pressure in cells. The answer is 1.",
        "short_answer": "1"
    },
    {
        "question": "Which of these would be the most effective way to reduce the amount of fossil fuel used for transportation?",
        "choices": ["build wider roads", "use public transportation", "increase parking spaces", "lower speed limits"],
        "cot_answer": "Building wider roads and increasing parking spaces would likely encourage more driving, thus more fossil fuel use. Lowering speed limits has a minor effect. Public transportation is the most effective because one bus or train replaces many individual cars, significantly reducing per-person fossil fuel consumption. The answer is 1.",
        "short_answer": "1"
    }
]

In [32]:
!mkdir -p data
# Download ARC-Challenge test set (first 50 examples)
from datasets import load_dataset
import json

arc_dataset = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
arc_data = []
for item in list(arc_dataset)[:50]:
    choices = item['choices']['text']
    labels = item['choices']['label']
    answer_key = item['answerKey']
    # Convert letter answer to index
    answer_idx = labels.index(answer_key) if answer_key in labels else ord(answer_key) - ord('A')
    arc_data.append({
        "question": item['question'],
        "choices": choices,
        "true_answer": str(answer_idx),
        "source": "ARC"
    })

with open("data/arc.jsonl", "w") as f:
    for item in arc_data:
        f.write(json.dumps(item) + "\n")

print(f"Loaded {len(arc_data)} ARC-Challenge questions")
print(json.dumps(arc_data[0], indent=2))

In [33]:
import json

def load_eval_data(task):
    with open(f"data/{task}.jsonl", "r") as f:
        return [json.loads(line) for line in f]

print("Example ARC question:")
print(json.dumps(load_eval_data("arc")[0], indent=4))

In [34]:
import os
import json
import datetime
from pathlib import Path
from tqdm import tqdm

def answer_questions(task, agent, action_type, answers_file):
    """Run agent on all questions in a task and save answers."""
    data = load_eval_data(task)
    Path("output").mkdir(exist_ok=True)
    
    existing = set()
    if os.path.exists(answers_file):
        with open(answers_file, "r") as f:
            for line in f:
                item = json.loads(line)
                existing.add(item["question"])
    
    for item in tqdm(data, desc=f"Running {action_type} on {task}"):
        q = item["question"]
        if isinstance(item.get("choices"), list):
            choices_str = "\n".join(f"  {i}. {c}" for i, c in enumerate(item["choices"]))
            q = f"{q}\nChoices:\n{choices_str}\nAnswer with only the choice number (0, 1, 2, ...)."
        if q in existing:
            continue
        try:
            answer = agent.run(q)
        except Exception as e:
            answer = f"ERROR: {e}"
        
        result = {
            "question": item["question"],
            "answer": answer,
            "true_answer": item["true_answer"],
            "source": item["source"],
            "model_id": model_llm.model_id,
            "agent_action_type": action_type,
            "timestamp": str(datetime.datetime.now())
        }
        with open(answers_file, "a") as f:
            f.write(json.dumps(result) + "\n")

### 2.1 Few-shot Direct Prompting


In [41]:
import importlib
import eval_utils
eval_utils = importlib.reload(eval_utils)
score_answers = eval_utils.score_answers

class FewShotReasoner:
    def __init__(self, model, n_shots):
        self.model = model
        self.n_shots = n_shots

    def build_input(self, question):
        """Build an n-shot direct prompt using ARC_EXAMPLARS."""
        prompt_lines = [
            "Answer the following questions. Choose the correct option by its number.",
            "",
        ]

        for ex in ARC_EXAMPLARS[:self.n_shots]:
            prompt_lines.append(f"Question: {ex['question']}")
            prompt_lines.append("Choices:")
            for i, choice in enumerate(ex["choices"]):
                prompt_lines.append(f"  {i}. {choice}")
            prompt_lines.append(f"Answer: {ex['short_answer']}")
            prompt_lines.append("")

        prompt_lines.append(f"Question: {question}")
        if "Choices:" not in question:
            prompt_lines.append("Choices:")
            prompt_lines.append("  0. [Option 0]")
            prompt_lines.append("  1. [Option 1]")
            prompt_lines.append("  2. [Option 2]")
            prompt_lines.append("  3. [Option 3]")
        prompt_lines.append("Answer:")
        return "\n".join(prompt_lines)

    def run(self, question):
        prompt = self.build_input(question)
        return self.model(prompt=prompt, max_tokens=32, temperature=0.0)


def run_arc_fewshot(task="arc", action_type="fewshot"):
    answers_file = f"output/{model_id.replace('/', '__')}__{action_type}__{task}.jsonl"
    reasoner = FewShotReasoner(model_llm, 8)
    answer_questions(task, reasoner, action_type, answers_file)
    df = score_answers([answers_file])
    print(df)

run_arc_fewshot()

### 2.2 Few-shot Chain-of-Thought Prompting


In [42]:
import re

class FewShotCoTReasoner:
    def __init__(self, model, n_shots):
        self.model = model
        self.n_shots = n_shots

    def build_input(self, question):
        """Build an n-shot chain-of-thought prompt using ARC_EXAMPLARS."""
        prompt_lines = [
            "Answer the following multiple-choice science questions.",
            "Show brief reasoning, then end with: The answer is X.",
            "",
        ]

        for ex in ARC_EXAMPLARS[:self.n_shots]:
            prompt_lines.append(f"Question: {ex['question']}")
            prompt_lines.append("Choices:")
            for i, choice in enumerate(ex["choices"]):
                prompt_lines.append(f"  {i}. {choice}")
            prompt_lines.append(f"Answer: {ex['cot_answer']}")
            prompt_lines.append("")

        prompt_lines.append(f"Question: {question}")
        if "Choices:" not in question:
            prompt_lines.append("Choices:")
            prompt_lines.append("  0. [Option 0]")
            prompt_lines.append("  1. [Option 1]")
            prompt_lines.append("  2. [Option 2]")
            prompt_lines.append("  3. [Option 3]")
        prompt_lines.append("Answer:")
        return "\n".join(prompt_lines)

    def extract_answer(self, response):
        """Extract the final numeric answer from CoT response."""
        # Prefer explicit pattern like "The answer is X"
        matches = re.findall(r"(?i)the\s+answer\s+is\s*[:\-]?\s*([0-9])", response)
        if matches:
            return matches[-1]

        # Fallback: last standalone digit in the response
        digits = re.findall(r"\b([0-9])\b", response)
        if digits:
            return digits[-1]

        # Final fallback: raw trimmed response
        return response.strip()

    def run(self, question):
        prompt = self.build_input(question)
        response = self.model(prompt=prompt, max_tokens=512, temperature=0.0)
        return self.extract_answer(response)


def run_arc_fewshot_cot(task="arc", action_type="fewshot_cot"):
    answers_file = f"output/{model_id.replace('/', '__')}__{action_type}__{task}.jsonl"
    reasoner = FewShotCoTReasoner(model_llm, 8)
    answer_questions(task, reasoner, action_type, answers_file)
    df = score_answers([answers_file])
    print(df)

run_arc_fewshot_cot()

In [54]:
from eval_utils import score_answers

fewshot_file = f"output/{model_id.replace('/', '__')}__fewshot__arc.jsonl"
fewshot_cot_file = f"output/{model_id.replace('/', '__')}__fewshot_cot__arc.jsonl"

df = score_answers([fewshot_file, fewshot_cot_file])
print(df)

---
## Building a ReAct Agent from Scratch 

- **MATH**: Math problems requiring precise calculation and reasoning
- **GAIA**: Complex research-oriented tasks requiring multiple capabilities (knowledge retrieval, calculation, and synthesis)

```

### 3.1 Implement Tools

Implement two tools that the agent will use.

In [43]:
!pip install -q wikipedia

In [44]:
import wikipedia

class WikiSearchTool:
    """Search Wikipedia and return a summary."""
    name = "wiki_search"
    description = "Search Wikipedia for a topic and return a summary. Input should be a search query string."

    def __call__(self, query: str) -> str:
        """Search Wikipedia and return the summary of the top result.
        Handle disambiguation and page not found errors gracefully."""
        query = (query or "").strip()
        if not query:
            return "Error: empty query."

        try:
            return wikipedia.summary(query, sentences=3, auto_suggest=False, redirect=True)
        except wikipedia.exceptions.DisambiguationError as e:
            options = e.options[:5] if e.options else []
            if options:
                return f"Disambiguation: {', '.join(options)}"
            return "Disambiguation: multiple pages found."
        except wikipedia.exceptions.PageError:
            try:
                candidates = wikipedia.search(query, results=5)
            except Exception:
                candidates = []

            if not candidates:
                return "No Wikipedia page found."

            title = candidates[0]
            try:
                return wikipedia.summary(title, sentences=3, auto_suggest=False, redirect=True)
            except Exception:
                return f"No Wikipedia page found for '{query}'. Try: {', '.join(candidates[:3])}"
        except Exception as e:
            return f"Wiki search error: {e}"


# Test
wiki_tool = WikiSearchTool()
print(wiki_tool("Eiffel Tower")[:500])

In [45]:
import ast
import math
import operator

class CalculatorTool:
    """Evaluate a mathematical expression."""
    name = "calculator"
    description = "Evaluate a mathematical expression. Input should be a valid Python math expression as a string."

    def __call__(self, expression: str) -> str:
        """Safely evaluate a math expression.
        Only allow basic math operations. Do NOT use eval() directly on untrusted input."""
        expression = (expression or "").strip()
        if not expression:
            return "Error: empty expression."

        allowed_binops = {
            ast.Add: operator.add,
            ast.Sub: operator.sub,
            ast.Mult: operator.mul,
            ast.Div: operator.truediv,
            ast.Pow: operator.pow,
            ast.Mod: operator.mod,
            ast.FloorDiv: operator.floordiv,
        }
        allowed_unaryops = {
            ast.UAdd: operator.pos,
            ast.USub: operator.neg,
        }
        allowed_funcs = {
            "sqrt": math.sqrt,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "log": math.log,
            "log10": math.log10,
            "exp": math.exp,
            "fabs": math.fabs,
            "ceil": math.ceil,
            "floor": math.floor,
            "factorial": math.factorial,
        }
        allowed_consts = {"pi": math.pi, "e": math.e}

        def _eval(node):
            if isinstance(node, ast.Expression):
                return _eval(node.body)
            if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
                return node.value
            if isinstance(node, ast.Num):
                return node.n
            if isinstance(node, ast.BinOp) and type(node.op) in allowed_binops:
                return allowed_binops[type(node.op)](_eval(node.left), _eval(node.right))
            if isinstance(node, ast.UnaryOp) and type(node.op) in allowed_unaryops:
                return allowed_unaryops[type(node.op)](_eval(node.operand))
            if isinstance(node, ast.Name) and node.id in allowed_consts:
                return allowed_consts[node.id]
            if isinstance(node, ast.Call):
                if isinstance(node.func, ast.Name) and node.func.id in allowed_funcs and len(node.keywords) == 0:
                    return allowed_funcs[node.func.id](*[_eval(arg) for arg in node.args])
                raise ValueError("Unsupported function call")
            raise ValueError("Unsupported expression")

        try:
            return str(_eval(ast.parse(expression, mode="eval")))
        except Exception as e:
            return f"Error: {e}"


# Test
calc_tool = CalculatorTool()
print(calc_tool("67905000 * 3"))
print(calc_tool("(8848.86 * 3.28084)"))
print(calc_tool("import os"))  # Should return an error message, not execute


### 3.2 Build the ReAct Agent

1. Construct a system prompt describing available tools and the ReAct format
2. Send the prompt to the LLM
3. Parse the LLM's output to extract `Thought`, `Action`, and `Action Input`
4. Execute the tool and append the `Observation` to the conversation
5. Repeat until the agent calls `finish` or reaches `max_steps`

In [46]:
class ReActAgent:
    def __init__(self, model, tools, max_steps=8):
        """
        Args:
            model: VLLMClient instance
            tools: list of tool instances (WikiSearchTool, CalculatorTool)
            max_steps: maximum number of Thought-Action-Observation cycles
        """
        self.model = model
        self.tools = {tool.name: tool for tool in tools}
        self.tools["finish"] = None  # special action to end
        self.max_steps = max_steps

    def build_system_prompt(self):
        """Build the system prompt describing tools and the ReAct format."""
        tool_lines = []
        for name, tool in self.tools.items():
            if name != "finish":
                tool_lines.append(f"- {name}: {tool.description}")

        return (
            "You are a ReAct agent. Solve the question using the ReAct loop.\n\n"
            "Available tools:\n"
            + "\n".join(tool_lines) + "\n"
            + "- finish: return the final answer and stop.\n\n"
            + "Use this exact format:\n"
            + "Thought: [reasoning]\n"
            + "Action: [tool_name or finish]\n"
            + "Action Input: [query or final answer]\n"
            + "Observation: [tool result]\n"
        )

    def parse_action(self, text):
        """Parse the LLM output to extract Action and Action Input.

        Returns:
            tuple: (action_name: str, action_input: str) or (None, None) if parsing fails
        """
        if not text:
            return None, None

        action = None
        action_input = None
        for line in text.splitlines():
            s = line.strip()
            if s.lower().startswith("action:") and action is None:
                action = s.split(":", 1)[1].strip().lower()
            elif s.lower().startswith("action input:") and action_input is None:
                action_input = s.split(":", 1)[1].strip()

        if action is None or action_input is None:
            return None, None
        return action, action_input

    def run(self, question):
        """Run the ReAct loop.

        Args:
            question: The user's question

        Returns:
            The final answer string, or None if max_steps reached
        """
        prompt = self.build_system_prompt() + f"\nQuestion: {question}\n"
        last_response = None

        for _ in range(self.max_steps):
            response = self.model(prompt=prompt, max_tokens=256, temperature=0.0).strip()
            last_response = response
            action, action_input = self.parse_action(response)

            if action is None:
                prompt += response + "\nObservation: Could not parse action. Follow exact format.\n"
                continue

            if action == "finish":
                return action_input

            tool = self.tools.get(action)
            if tool is None:
                observation = f"Unknown action: {action}"
            else:
                try:
                    observation = tool(action_input)
                except Exception as e:
                    observation = f"Tool error: {e}"

            prompt += response + f"\nObservation: {observation}\n"

        return last_response if last_response is not None else None

In [47]:
# Test the agent on a few examples
agent = ReActAgent(
    model=model_llm,
    tools=[WikiSearchTool(), CalculatorTool()],
    max_steps=8
)

# Simple test
test_questions = [
    "What is the population of France multiplied by 3?",
    "Who wrote the novel '1984', and in what year did that author die?",
    "What is the factorial of 7?",
]

for q in test_questions:
    print(f"Q: {q}")
    answer = agent.run(q)
    print(f"A: {answer}")
    print("-" * 50)

### 3.3 Baseline: Vanilla Chat Agent

For comparison, a simple chat agent with no tools.

In [48]:
class ChatAgent:
    """Simple agent that directly asks the LLM without any tools."""
    def __init__(self, model):
        self.model = model
        self.tokenizer = AutoTokenizer.from_pretrained(model.model_id)

    def run(self, task):
        messages = [{"role": "user", "content": task + "\nAnswer concisely."}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        return self.model(prompt=prompt, max_tokens=256)

### 3.4 Evaluation

vanilla ChatAgent and ReActAgent on MATH and GAIA.

In [49]:
!mkdir -p data
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/ranpox/comp3361-spring2025/refs/heads/main/assignments/A3/data"
for dataset_name in ["math", "gaia"]:
    urllib.request.urlretrieve(f"{BASE_URL}/{dataset_name}.jsonl", f"data/{dataset_name}.jsonl")
    print(f"Downloaded {dataset_name}.jsonl")

In [50]:
# Run agents and save answers to files
all_answer_files = []

for task in ["math", "gaia"]:
    # Vanilla ChatAgent
    vanilla_file = f"output/{model_id.replace('/', '__')}__vanilla__{task}.jsonl"
    chat_agent = ChatAgent(model_llm)
    answer_questions(task, chat_agent, "vanilla", vanilla_file)
    all_answer_files.append(vanilla_file)

    # ReAct Agent
    react_file = f"output/{model_id.replace('/', '__')}__react__{task}.jsonl"
    react_agent = ReActAgent(
        model=model_llm,
        tools=[WikiSearchTool(), CalculatorTool()],
        max_steps=8
    )
    answer_questions(task, react_agent, "react", react_file)
    all_answer_files.append(react_file)

print("All agents finished. Answer files:", all_answer_files)

In [51]:
# Score answers and display results
from eval_utils import score_answers

df = score_answers(all_answer_files)
print(df.to_string())

---
## Summary Table

Fill in your results:

| Section | Task | Method | Metric | Score |
|---------|------|--------|--------|-------|
| 1 | Decoding | Greedy | PPL / Fluency / Diversity / Repetition | 1.92 / 1.0000 / 0.0449 / 0.7545| 
| 1 | Decoding | Vanilla Sampling | PPL / Fluency / Diversity / Repetition | 50.08 / 0.2800 / 0.9921 / 0.0053|
| 1 | Decoding | Temperature (0.8) | PPL / Fluency / Diversity / Repetition | 13.98 / 0.8000 / 0.9769 / 0.0033|
| 1 | Decoding | Top-k (20) | PPL / Fluency / Diversity / Repetition | 11.69 / 0.8800 / 0.9694 / 0.0060|
| 1 | Decoding | Top-p (0.7) | PPL / Fluency / Diversity / Repetition | 11.53 / 0.8000 / 0.9604 / 0.0185|
| 2 | ARC-Challenge | Few-shot Direct | Accuracy | 0.44|
| 2 | ARC-Challenge | Few-shot CoT | Accuracy | 0.42|
| 3 | MATH | Vanilla / ReAct | Accuracy | 0.22 / 0.36|
| 3 | GAIA | Vanilla / ReAct | Accuracy | 0.0625 / 0.0625|